# 04 · Neural baseline trained from scratch: BiLSTM-CTC

**Question.** How far does a conventional neural ASR model get using DhoNam alone, with no pre-training?
This is the reference that tells us how much multilingual self-supervised pre-training (notebook 05) is worth.

**Architecture.** 80-dim log-mel (25 ms / 10 ms) → per-utterance mean/variance normalisation → SpecAugment
(train only) → 2×Conv2d (stride 2, 4× time subsampling) → 3-layer BiLSTM (320 units/direction) → linear →
CTC over the character vocabulary built from training transcripts. All settings live in
`configs/experiments.yaml` (`bilstm_ctc`).

Kaggle: GPU T4 x1. Input: `splits.csv` (notebook 02).

In [ ]:
# ── Setup: locate the project source (src/dholuo_asr) and the output folder ──────────
import os, sys, subprocess, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# After pushing the repo to GitHub, put its URL here (Kaggle: Settings → Internet ON).
# Alternatively upload the repo as a private Kaggle dataset and attach it as an input.
REPO_URL = os.environ.get("DHOLUO_REPO_URL", "https://github.com/YOUR-USERNAME/dholuo-asr.git")

def _find_src(max_depth=4):
    for root in [Path.cwd(), Path.cwd().parent, Path("/kaggle/working"), Path("/kaggle/input")]:
        if not root.exists():
            continue
        base = len(root.parts)
        for dirpath, dirnames, filenames in os.walk(root):
            if len(Path(dirpath).parts) - base >= max_depth or len(filenames) > 500:
                dirnames[:] = []          # don't descend into audio folders
            if Path(dirpath).name == "dholuo_asr" and "__init__.py" in filenames:
                return Path(dirpath).parent
    return None

SRC = _find_src()
if SRC is None and "YOUR-USERNAME" not in REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/kaggle/working/dholuo-asr"], check=True)
    SRC = Path("/kaggle/working/dholuo-asr/src")
if SRC is None:
    raise RuntimeError("Project source not found: set REPO_URL (Internet ON) or attach the repo as a dataset.")
sys.path.insert(0, str(SRC))
REPO = SRC.parent
ON_KAGGLE = Path("/kaggle/working").exists()
OUT = Path(os.environ.get("DHOLUO_OUT", "/kaggle/working" if ON_KAGGLE else REPO / "outputs"))
OUT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOTS = [OUT, Path("/kaggle/input")]
print("source :", SRC)
print("outputs:", OUT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})
DIALECT_COLORS = {"Milambo": "#2a6f97", "Nyanduat": "#e07a5f"}
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG / f"{name}.png", bbox_inches="tight"); plt.show()

In [ ]:
EXPERIMENT = os.environ.get("EXPERIMENT", "bilstm_ctc")
OVERRIDES = {}

In [ ]:
import torch
from dholuo_asr.utils import find_artifact, load_experiment, dump_json
from dholuo_asr.runner import run_experiment
CONFIG = REPO / "configs" / "experiments.yaml"
cfg = load_experiment(CONFIG, EXPERIMENT)
for k, v in OVERRIDES.items():          # e.g. {"train.max_steps": 3000}
    sect, key = k.split(".")
    cfg[sect][key] = v
print(json.dumps(cfg, indent=1))
splits = pd.read_csv(find_artifact("splits.csv", ARTIFACT_ROOTS), keep_default_na=False, na_values=[""])
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
metrics = run_experiment(cfg, splits, OUT / "results", log=print)

### Training curve (validation WER is the model-selection criterion; the test set is never consulted)

In [ ]:
RUN = OUT / "results" / cfg["name"]
log = pd.read_csv(RUN / "train_log.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(log["step"], log["train_loss"], marker="o"); ax[0].set_title("training CTC loss"); ax[0].set_xlabel("step")
ax[1].plot(log["step"], log["val_wer"], marker="o", label="val WER"); ax[1].plot(log["step"], log["val_cer"], marker="o", label="val CER")
ax[1].axvline(metrics["best"]["step"], ls="--", c="k", lw=1, label="selected checkpoint"); ax[1].legend(); ax[1].set_xlabel("step")
savefig(f"curve_{cfg['name']}")
summary = {s: {k: metrics[s][k] for k in ["wer", "cer", "wer_ci", "cer_ci", "n_utts"]} for s in ["val", "test"]}
print(json.dumps({"best": metrics["best"], "params_trainable": metrics["params_trainable"], **summary}, indent=1))
pd.read_csv(RUN / "predictions_test.csv")[["dialect", "ref", "hyp"]].sample(10, random_state=0)